# MRS Database Systematic Audit

This notebook works through every source file, loads what it contains, cross-matches spectra and labels across files, and produces a final audit table showing:
- Which files were used and how
- Which records overlapped between files (exact or near duplicates)
- Which records are unique to a single file
- Which records remain unlabelled and why

**Run cells top to bottom.** All results accumulate into a master `registry` DataFrame. The final cells write summary CSVs and print the file-level audit.

---

## File inventory

| # | File | Expected contents |
|---|------|-------------------|
| F1 | `Multiclass_RMSE_QUEST.mat` | All spectra + quality labels (short+long TE, eTumour+INTERPRET) |
| F2 | `Matriz_originales.xml` | eTumour short-TE spectra + tissue labels |
| F3 | `1TE-ShortMRS_corregido_.xml` | INTERPRET short-TE spectra + tissue labels |
| F4 | `1TE-LongMRS_corregido_.xml` | INTERPRET long-TE spectra + tissue labels |
| F5 | `Listado_etumour2.txt` | eTumour case → WHO pathology text |
| F6 | `Listado_etumour-SNR-WBW.txt` | eTumour per-spectrum SNR + WBW |
| F7 | `Nocaso_patologÍa_base_de_datoS.xls` | Additional pathology DB (inspect only) |
| F8 | `Tablas_consensus_vs_original_pathology.xls` | Consensus vs original label comparison (inspect only) |
| F9 | `Matriz_Reconstruido_K20_17_2018.xml` | Reconstructed spectra K=20 (artefact removed) |
| F10 | `Matriz_Reconstruidos_K5_16_2018_1.xml` | Reconstructed spectra K=5 |
| F11 | `Matriz_Reconstruidos_K20_8sources.xml` | Reconstructed spectra K=20 8-source |
| F12 | `short.mat` | Short-TE spectra only (no metadata) |
| F13 | `Long_echo_data.mat` | Long-TE spectra only (no metadata) |
| F14 | `label.mat` | Quality labels for short.mat |
| F15 | `Long_echo_labels.mat` | Quality labels for Long_echo_data.mat |
| F16 | `convex_nmf.py` | Algorithm source (Python 2 — not a data file) |

---
## 0. Configuration and shared utilities

In [6]:
import numpy as np
import pandas as pd
import scipy.io as sio
import xml.etree.ElementTree as ET
import re
import os
from pathlib import Path
from collections import defaultdict, Counter

# ── CONFIGURE: set this to wherever your files live ──────────────────────
DATA_DIR = Path("./data/")
# ─────────────────────────────────────────────────────────────────────────

# All expected files and their short labels
FILES = {
    "F1":  "Matrices_y_datos_tesis_Yeni/Multiclass_RMSE_QUEST.mat",
    "F2":  "Matrices_Yeni_XML/Matriz_originales.xml",
    "F3":  "Matrices_y_datos_tesis_Yeni/1TE-ShortMRS(corregido).xml",
    "F4":  "Matrices_y_datos_tesis_Yeni/1TE-LongMRS(corregido).xml",
    "F5":  "Listado_etumour2.txt",
    "F6":  "Listado_etumour-SNR-WBW.txt",
    "F7":  "Nocaso_patologÍa_base_de_datoS.xls",
    "F8":  "Tablas_consensus_vs_original_pathology.xls",
    "F9":  "Matrices_Yeni_XML/Matriz_Reconstruido_K20_17_2018.xml",
    "F10": "Matrices_Yeni_XML/Matriz_Reconstruidos_K5_16_2018_1.xml",
    "F11": "Matrices_Yeni_XML/Matriz_Reconstruidos_K20_8sources.xml",
    "F12": "Matrices_y_datos_tesis_Yeni/short.mat",
    "F13": "Matrices_y_datos_tesis_Yeni/Long_echo_data.mat",
    "F14": "Matrices_y_datos_tesis_Yeni/label.mat",
    "F15": "Matrices_y_datos_tesis_Yeni/Long_echo_labels.mat",
    "F16": "Matrices_y_datos_tesis_Yeni/convex_nmf.py",
}

# Shared constants
INTERPRET_OFFSET = 777_770_000
PPM = np.linspace(-2.7, 7.1, 512)

# MSE thresholds for spectrum matching
THRESH_EXACT  = 1e-4   # treated as identical spectrum
THRESH_NEAR   = 0.5    # same case, different preprocessing

WHO_TO_TISSUE = {
    "GLIOBLASTOMA": "gl", "ANAPLASTIC ASTROCYTOMA": "a3",
    "ASTROCYTOMA": "a2", "FIBRILLARY ASTROCYTOMA": "a2",
    "PILOCYTIC ASTROCYTOMA": "pi", "OLIGODENDROGLIOMA": "od",
    "ANAPLASTIC OLIGODENDROGLIOMA": "od", "OLIGOASTROCYTOMA": "oa",
    "ANAPLASTIC OLIGOASTROCYTOMA": "oa", "MENINGIOMA": "me",
    "MENINGOTHELIAL MENINGIOMA": "me", "FIBROUS MENINGIOMA": "me",
    "TRANSITIONAL MENINGIOMA": "me", "PSAMMOMATOUS MENINGIOMA": "me",
    "SECRETORY MENINGIOMA": "me", "MICROCYSTIC MENINGIOMA": "me",
    "CLEAR CELL MENINGIOMA": "me", "ATYPICAL MENINGIOMA": "me",
    "METASTASIS": "mm", "MALIGNANT LYMPHOMAS": "ly", "LYMPHOMA": "ly",
    "PITUITARY ADENOMA": "pn", "MEDULLOBLASTOMA": "mb",
    "EPENDYMOMA": "ep", "ANAPLASTIC EPENDYMOMA": "ep",
    "TANYCYTIC EPENDYMOMA": "ep", "SCHWANNOMA": "sc",
    "HAEMANGIOBLASTOMA": "hb", "HAEMANGIOPERICYTOMA": "hb",
    "CRANIOPHARYNGIOMA": "cp", "ADAMANTINOMATOUS CRANIOPHARYNGIOMA": "cp",
    "GANGLIOGLIOMA": "gg", "GANGLIOCYTOMA": "gg",
    "CHOROID PLEXUS PAPILLOMA": "cpp", "NEUROBLASTOMA": "nb",
    "NORMAL TISSUE": "no", "ABSCESS": "ab",
    "DYSEMBRYOPLASTIC NEUROEPITHELIAL TUMOUR": "dnet",
    "PINEOBLASTOMA": "pb", "PINEOCYTOMA": "pc",
    "ATYPICAL TERATOID/RHABDOID TUMOUR": "at",
    "GLIOSARCOMA": "gs", "GLIOMATOSIS CEREBRI": "gc",
    "GERMINOMA": "ge", "MIXED GERM CELL TUMOURS": "ge",
    "TERATOMA": "te", "CARCINOMA": "mm",
    "RADIATION NECROSIS": "ra", "CENTRAL NEUROCYTOMA": "cn",
    "SOLITARY FIBROUS TUMOUR": "sf", "MELANOCYTOMA": "ml",
    "MALIGNANT MELANOMA": "ml", "HAEMANGIOMA": "hm",
    "CHORDOID GLIOMA OF THE 3RD VENTRICLE": "cg",
    "SUPRATENTORIAL PRIMITIVE NEUROECTODERMAL TUMOUR": "pnet",
}

def who_to_code(text):
    """Map a WHO pathology string to a short tissue code. Returns None if unmappable."""
    if not text or str(text).strip() in ("", "NA", "0", "NULL", "null", "OTHER"):
        return None
    clean = re.sub(r"\d{4}/\d", "", str(text)).strip().upper().strip("'\"")
    if clean in WHO_TO_TISSUE:
        return WHO_TO_TISSUE[clean]
    for key, code in WHO_TO_TISSUE.items():
        if key in clean:
            return code
    return None

def parse_xml_cases(path):
    """
    Parse an MRS XML file.
    Returns a list of dicts: {case_id, tissue_code, spectrum (np.float32 array)}.
    """
    tree = ET.parse(path)
    records = []
    for case in tree.getroot().findall("Case"):
        tissue = case.find("Tissue")
        pts_el = case.find(".//Points")
        if pts_el is None:
            continue
        records.append({
            "case_id":     case.get("ID"),
            "tissue_code": tissue.get("Type", "").strip() if tissue is not None else "",
            "spectrum":    np.array(pts_el.text.strip().split(), dtype=np.float32),
        })
    return records

def spectrum_matrix(records):
    """Stack spectra from a list of parse_xml_cases records into (N, 512)."""
    return np.vstack([r["spectrum"] for r in records])

def find_best_match(query_spec, ref_matrix):
    """
    Find the closest row in ref_matrix to query_spec by MSE.
    Returns (best_index, best_mse).
    """
    mse = np.mean((ref_matrix - query_spec) ** 2, axis=1)
    idx = mse.argmin()
    return int(idx), float(mse[idx])

# ── File presence check ───────────────────────────────────────────────────
print("File presence check:")
for fid, fname in FILES.items():
    path = DATA_DIR / fname
    size = f"{path.stat().st_size / 1e6:.1f} MB" if path.exists() else "MISSING"
    status = "✓" if path.exists() else "✗ MISSING"
    print(f"  {fid:4s}  {status}  {size:10s}  {fname}")

# Global registry: one row per spectrum across all sources
# Populated incrementally by each section below
registry = []  # list of dicts, converted to DataFrame at the end
file_usage = {fid: {"loaded": False, "records": 0, "role": "", "notes": ""}
              for fid in FILES}

print("\nSetup complete.")

File presence check:
  F1    ✓  7.8 MB      Matrices_y_datos_tesis_Yeni/Multiclass_RMSE_QUEST.mat
  F2    ✓  4.7 MB      Matrices_Yeni_XML/Matriz_originales.xml
  F3    ✓  1.5 MB      Matrices_y_datos_tesis_Yeni/1TE-ShortMRS(corregido).xml
  F4    ✓  1.5 MB      Matrices_y_datos_tesis_Yeni/1TE-LongMRS(corregido).xml
  F5    ✓  0.2 MB      Listado_etumour2.txt
  F6    ✓  0.2 MB      Listado_etumour-SNR-WBW.txt
  F7    ✓  0.4 MB      Nocaso_patologÍa_base_de_datoS.xls
  F8    ✓  0.2 MB      Tablas_consensus_vs_original_pathology.xls
  F9    ✓  3.3 MB      Matrices_Yeni_XML/Matriz_Reconstruido_K20_17_2018.xml
  F10   ✓  3.8 MB      Matrices_Yeni_XML/Matriz_Reconstruidos_K5_16_2018_1.xml
  F11   ✓  3.3 MB      Matrices_Yeni_XML/Matriz_Reconstruidos_K20_8sources.xml
  F12   ✓  4.2 MB      Matrices_y_datos_tesis_Yeni/short.mat
  F13   ✓  3.5 MB      Matrices_y_datos_tesis_Yeni/Long_echo_data.mat
  F14   ✓  0.0 MB      Matrices_y_datos_tesis_Yeni/label.mat
  F15   ✓  0.0 MB      Matrices_y_da

---
## 1.  F1 — `Multiclass_RMSE_QUEST.mat`

The hub file. Contains all spectra with quality labels. We extract every row and register it. Tissue labels are added later by cross-matching against the other files.

**517-column layout**: `[case_id, svs_code, exp_idx, SNR_mat, WBW_mat, spectrum×512]`  
**Case ID encoding**: eTumour rows have `col0 < 100 000`; INTERPRET rows have `col0 = 777_770_000 + N`.

In [7]:
mat = sio.loadmat(DATA_DIR / FILES["F1"])
file_usage["F1"]["loaded"] = True

def extract_mat_block(data, labels, rmse_mat, echo_time):
    """
    Turn one TE block from the MAT file into a list of registry dicts.
    Each dict has a unique row_uid and enough metadata to join later.
    """
    flat_labels = np.array([str(l.flat[0]) for l in labels.flatten()])
    rows = []
    for i in range(len(data)):
        raw_id = int(data[i, 0])
        is_interp = raw_id > 100_000
        case_num  = raw_id - INTERPRET_OFFSET if is_interp else raw_id
        case_id   = f"I{int(case_num):04d}" if is_interp else f"et{raw_id}"
        q = flat_labels[i]
        rows.append({
            # identity
            "row_uid":        f"{case_id}__{echo_time}__{int(data[i, 2])}",
            "case_id":        case_id,
            "dataset":        "INTERPRET" if is_interp else "eTumour",
            "echo_time":      echo_time,
            "exp_idx":        int(data[i, 2]),
            # quality
            "quality":        q,
            "quality_code":   {"Good": 0, "Poor": 1, "Bad": 2}.get(q, -1),
            "is_artefactual": int(q != "Good"),
            "quest_rmse":     float(rmse_mat[i, 3]),
            # acquisition metrics (MAT version — SNR definition differs from DB)
            "snr_mat":        float(data[i, 3]),
            "wbw_mat":        float(data[i, 4]),
            # spectrum (stored as list to allow DataFrame)
            "spectrum":       data[i, 5:].astype(np.float32),
            # label placeholders — filled in later sections
            "tissue_code":    None,
            "tissue_source":  None,
            "who_consensus":  None,
            "who_original":   None,
            "snr_db":         None,   # from F6
            "wbw_hz":         None,   # from F6
            # audit fields — filled in later
            "f1_row":         i,
            "source_files":   ["F1"],
            "xml_match_file": None,
            "xml_match_id":   None,
            "xml_match_mse":  None,
        })
    return rows

short_rows = extract_mat_block(
    mat["ShortTE"], mat["ShortTE_label"], mat["RMSE_ShortTE"], "short"
)
long_rows = extract_mat_block(
    mat["LongTE"],  mat["LongTE_label"],  mat["RMSE_LongTE"],  "long"
)

registry = short_rows + long_rows
file_usage["F1"]["records"] = len(registry)
file_usage["F1"]["role"]    = "Primary spectrum + quality label source"

print(f"F1 loaded: {len(short_rows)} short-TE + {len(long_rows)} long-TE = {len(registry)} total rows")
print(f"  eTumour short: {sum(1 for r in short_rows if r['dataset']=='eTumour')}")
print(f"  INTERPRET short: {sum(1 for r in short_rows if r['dataset']=='INTERPRET')}")
print(f"  eTumour long: {sum(1 for r in long_rows if r['dataset']=='eTumour')}")
print(f"  INTERPRET long: {sum(1 for r in long_rows if r['dataset']=='INTERPRET')}")

# Index for fast lookup: row_uid -> position in registry
uid_to_idx = {r["row_uid"]: i for i, r in enumerate(registry)}

F1 loaded: 1180 short-TE + 977 long-TE = 2157 total rows
  eTumour short: 750
  INTERPRET short: 430
  eTumour long: 621
  INTERPRET long: 356


---
## 2.  F2 — `Matriz_originales.xml` (eTumour, tissue labels)

Strategy: for each XML spectrum, compute MSE against every eTumour short-TE MAT row. If MSE < `THRESH_EXACT`, the MAT row gets the XML tissue label. Records the match MSE and the XML case ID so you can inspect borderline cases.

In [8]:
f2_records = parse_xml_cases(DATA_DIR / FILES["F2"])
f2_matrix  = spectrum_matrix(f2_records)
file_usage["F2"]["loaded"]  = True
file_usage["F2"]["records"] = len(f2_records)
file_usage["F2"]["role"]    = "eTumour tissue labels via exact spectrum match"

print(f"F2 loaded: {len(f2_records)} spectra")
print(f"  Tissue distribution: {dict(Counter(r['tissue_code'] for r in f2_records).most_common())}")

# Match each F2 spectrum against F1 eTumour short-TE rows
et_short_indices = [
    i for i, r in enumerate(registry)
    if r["dataset"] == "eTumour" and r["echo_time"] == "short"
]
et_short_matrix = np.vstack([registry[i]["spectrum"] for i in et_short_indices])

f2_matched   = 0
f2_unmatched = 0
f2_used_rows = set()   # which XML rows were matched

for xi, xrec in enumerate(f2_records):
    mse_vec = np.mean((et_short_matrix - xrec["spectrum"]) ** 2, axis=1)
    best_local = int(mse_vec.argmin())
    best_mse   = float(mse_vec[best_local])
    reg_idx    = et_short_indices[best_local]

    if best_mse < THRESH_EXACT:
        r = registry[reg_idx]
        if r["tissue_code"] is None:   # don't overwrite an already-assigned label
            r["tissue_code"]   = xrec["tissue_code"]
            r["tissue_source"] = "F2_exact_match"
            r["xml_match_file"]= "F2"
            r["xml_match_id"]  = xrec["case_id"]
            r["xml_match_mse"] = best_mse
        if "F2" not in r["source_files"]:
            r["source_files"].append("F2")
        f2_matched += 1
        f2_used_rows.add(xi)
    else:
        f2_unmatched += 1

# Near-match pass: spectra that didn't hit EXACT — record near-match info for inspection
near_match_records = []
for xi, xrec in enumerate(f2_records):
    if xi in f2_used_rows:
        continue
    mse_vec  = np.mean((et_short_matrix - xrec["spectrum"]) ** 2, axis=1)
    best_local = int(mse_vec.argmin())
    best_mse   = float(mse_vec[best_local])
    reg_idx    = et_short_indices[best_local]
    if best_mse < THRESH_NEAR:
        near_match_records.append({
            "xml_id":      xrec["case_id"],
            "xml_tissue":  xrec["tissue_code"],
            "mat_case_id": registry[reg_idx]["case_id"],
            "mse":         best_mse,
            "verdict":     "near_match_different_preprocessing",
        })

file_usage["F2"]["notes"] = (
    f"{f2_matched} XML spectra exactly matched to F1 rows; "
    f"{f2_unmatched} XML spectra unmatched (different preprocessing); "
    f"{len(near_match_records)} near-matches (0 < MSE < {THRESH_NEAR})"
)

print(f"\nF2 matching results:")
print(f"  Exact matches (MSE < {THRESH_EXACT}): {f2_matched}")
print(f"  Near matches  (MSE < {THRESH_NEAR}):  {len(near_match_records)}")
print(f"  Unmatched XML rows:                   {f2_unmatched}")

# Show unmatched XML rows — these are F2 spectra that don't appear in F1
f2_unmatched_ids = [f2_records[xi]["case_id"] for xi in range(len(f2_records)) if xi not in f2_used_rows]
print(f"\n  F2 rows not matched to any F1 entry: {len(f2_unmatched_ids)}")
print(f"  Sample: {f2_unmatched_ids[:10]}")

# How many F1 eTumour short rows now have tissue?
labelled_now = sum(1 for i in et_short_indices if registry[i]["tissue_code"] is not None)
print(f"\n  F1 eTumour short-TE rows labelled after F2: {labelled_now}/{len(et_short_indices)}")

F2 loaded: 777 spectra
  Tissue distribution: {'gl': 294, 'mm': 122, 'me': 116, 'a2': 103, 'od': 44, 'pi': 43, 'no': 27, 'oa': 23, '': 4, 'bg': 1}

F2 matching results:
  Exact matches (MSE < 0.0001): 535
  Near matches  (MSE < 0.5):  19
  Unmatched XML rows:                   242

  F2 rows not matched to any F1 entry: 242
  Sample: ['S1gl(Ori_827.txt)', 'S1gl(Ori_877.txt)', 'S1gl(Ori_968.txt)', 'S1gl(Ori_750.txt)', 'S1gl(Ori_752.txt)', 'S1gl(Ori_753.txt)', 'S1a2(Ori_754.txt)', 'S1gl(Ori_755.txt)', 'S1me(Ori_759.txt)', 'S1mm(Ori_761.txt)']

  F1 eTumour short-TE rows labelled after F2: 528/750


---
## 3.  F3 + F4 — INTERPRET XMLs (tissue labels)

Both XMLs carry tissue labels keyed by case number (`I0001` → 1). We build a combined lookup, then assign tissue to all INTERPRET MAT rows whose case number appears in either XML.

We also check tissue agreement between F3 and F4 for cases present in both, and run a spectrum-level match for unresolved INTERPRET MAT rows against both XML spectrum sets.

In [9]:
f3_records = parse_xml_cases(DATA_DIR / FILES["F3"])
f4_records = parse_xml_cases(DATA_DIR / FILES["F4"])
f3_matrix  = spectrum_matrix(f3_records)
f4_matrix  = spectrum_matrix(f4_records)

file_usage["F3"].update({"loaded": True, "records": len(f3_records),
    "role": "INTERPRET tissue labels (short-TE XML)"})
file_usage["F4"].update({"loaded": True, "records": len(f4_records),
    "role": "INTERPRET tissue labels (long-TE XML)"})

# Build case-number → tissue lookup from both XMLs
# Decode 'I0001' → 1
f3_tissue = {int(r["case_id"][1:]): r["tissue_code"] for r in f3_records}
f4_tissue = {int(r["case_id"][1:]): r["tissue_code"] for r in f4_records}

print(f"F3 (short XML): {len(f3_tissue)} cases, nums {min(f3_tissue)}-{max(f3_tissue)}")
print(f"F4 (long XML):  {len(f4_tissue)} cases, nums {min(f4_tissue)}-{max(f4_tissue)}")

only_in_f3 = set(f3_tissue) - set(f4_tissue)
only_in_f4 = set(f4_tissue) - set(f3_tissue)
in_both    = set(f3_tissue) & set(f4_tissue)
print(f"  Only in F3: {len(only_in_f3)}  |  Only in F4: {len(only_in_f4)}  |  In both: {len(in_both)}")

# Cross-validate tissue agreement where case appears in both
tissue_disagreements = [
    {"case_num": n, "f3_tissue": f3_tissue[n], "f4_tissue": f4_tissue[n]}
    for n in in_both if f3_tissue[n] != f4_tissue[n]
]
print(f"  Tissue disagreements between F3 and F4: {len(tissue_disagreements)}")
if tissue_disagreements:
    print("  Disagreements:")
    for d in tissue_disagreements:
        print(f"    I{d['case_num']:04d}: F3={d['f3_tissue']}, F4={d['f4_tissue']}")

# Combined lookup: F3 takes priority (it's the superset for new case numbers)
interp_tissue_lookup = {**f4_tissue, **f3_tissue}  # F3 overwrites F4 where both present
print(f"\nCombined INTERPRET tissue lookup: {len(interp_tissue_lookup)} unique case numbers")

# ── Step A: assign tissue to INTERPRET MAT rows by case number ────────────
interp_indices = [
    i for i, r in enumerate(registry) if r["dataset"] == "INTERPRET"
]
assigned_by_id = 0
for i in interp_indices:
    r = registry[i]
    case_num = int(r["case_id"][1:])  # I0098 → 98
    tissue = interp_tissue_lookup.get(case_num)
    if tissue and r["tissue_code"] is None:
        r["tissue_code"]    = tissue
        r["tissue_source"]  = "F3_or_F4_id_match"
        # record which file it came from
        if case_num in f3_tissue:
            r["xml_match_file"] = "F3"
            if "F3" not in r["source_files"]: r["source_files"].append("F3")
        if case_num in f4_tissue:
            if "F4" not in r["source_files"]: r["source_files"].append("F4")
            if r["xml_match_file"] is None: r["xml_match_file"] = "F4"
        r["xml_match_id"]  = f"I{case_num:04d}"
        r["xml_match_mse"] = 0.0  # ID match, no MSE
        assigned_by_id += 1

print(f"\nAssigned tissue to {assigned_by_id} INTERPRET rows via case ID")

# ── Step B: spectrum-level match for still-unresolved INTERPRET rows ───────
# Build matrices for F3 and F4 spectra indexed by case number
f3_spec_by_num = {int(r["case_id"][1:]): r["spectrum"] for r in f3_records}
f4_spec_by_num = {int(r["case_id"][1:]): r["spectrum"] for r in f4_records}

unresolved_interp = [
    i for i in interp_indices if registry[i]["tissue_code"] is None
]
print(f"Unresolved INTERPRET rows after ID match: {len(unresolved_interp)}")
print(f"Attempting spectrum-level match against F3 and F4...")

spec_match_results = []
for i in unresolved_interp:
    r      = registry[i]
    q_spec = r["spectrum"]
    row_te = r["echo_time"]

    # Match against F3 (short XML) and F4 (long XML)
    f3_idx, f3_mse = find_best_match(q_spec, f3_matrix)
    f4_idx, f4_mse = find_best_match(q_spec, f4_matrix)

    best_mse  = min(f3_mse, f4_mse)
    best_file = "F3" if f3_mse <= f4_mse else "F4"
    best_idx  = f3_idx if best_file == "F3" else f4_idx
    best_recs = f3_records if best_file == "F3" else f4_records
    best_rec  = best_recs[best_idx]

    verdict = (
        "exact_match" if best_mse < THRESH_EXACT
        else "near_match" if best_mse < THRESH_NEAR
        else "no_match"
    )
    spec_match_results.append({
        "reg_idx":      i,
        "mat_case_id":  r["case_id"],
        "echo_time":    row_te,
        "quality":      r["quality"],
        "snr_mat":      r["snr_mat"],
        "f3_best_id":   f3_records[f3_idx]["case_id"],
        "f3_mse":       round(f3_mse, 6),
        "f4_best_id":   f4_records[f4_idx]["case_id"],
        "f4_mse":       round(f4_mse, 6),
        "best_file":    best_file,
        "best_xml_id":  best_rec["case_id"],
        "best_tissue":  best_rec["tissue_code"],
        "best_mse":     round(best_mse, 6),
        "verdict":      verdict,
    })

    # If exact match, assign tissue
    if verdict == "exact_match" and r["tissue_code"] is None:
        r["tissue_code"]    = best_rec["tissue_code"]
        r["tissue_source"]  = f"{best_file}_spectrum_match"
        r["xml_match_file"] = best_file
        r["xml_match_id"]   = best_rec["case_id"]
        r["xml_match_mse"]  = best_mse
        if best_file not in r["source_files"]: r["source_files"].append(best_file)

spec_match_df = pd.DataFrame(spec_match_results)
verdicts = spec_match_df["verdict"].value_counts() if len(spec_match_df) else {}
print(f"  Spectrum match verdicts: {dict(verdicts)}")

# Store for inspection
interp_spec_match_df = spec_match_df.copy()

file_usage["F3"]["notes"] = f"{len(f3_tissue)} tissue labels; {len(only_in_f3)} cases unique to F3"
file_usage["F4"]["notes"] = f"{len(f4_tissue)} tissue labels; {len(only_in_f4)} cases unique to F4 (=0 — F4 is subset of F3)"

# Inspection table: all spectrum-match results for unresolved INTERPRET rows
print(f"\nFull spectrum match table for unresolved INTERPRET rows:")
display_cols = ["mat_case_id","echo_time","quality","snr_mat",
                "f3_best_id","f3_mse","f4_best_id","f4_mse","verdict"]
print(spec_match_df[display_cols].to_string(index=False) if len(spec_match_df) else "(none)")

F3 (short XML): 304 cases, nums 1-1476
F4 (long XML):  266 cases, nums 1-1476
  Only in F3: 38  |  Only in F4: 0  |  In both: 266
  Tissue disagreements between F3 and F4: 0

Combined INTERPRET tissue lookup: 304 unique case numbers

Assigned tissue to 538 INTERPRET rows via case ID
Unresolved INTERPRET rows after ID match: 248
Attempting spectrum-level match against F3 and F4...
  Spectrum match verdicts: {'no_match': np.int64(246), 'near_match': np.int64(2)}

Full spectrum match table for unresolved INTERPRET rows:
mat_case_id echo_time quality   snr_mat f3_best_id    f3_mse f4_best_id    f4_mse    verdict
      I0099     short    Good  9.721622      I0173  1.809381      I1036  5.656154   no_match
      I0118     short    Good  9.525100      I1080  1.254235      I1056  9.670527   no_match
      I0119     short    Good  6.866806      I0155 11.957273      I1260  6.909705   no_match
      I0119     short    Good  5.175614      I0332  3.826255      I1036  8.140420   no_match
      I0132 

---
## 4.  F5 — `Listado_etumour2.txt` (eTumour pathology labels)

Joins by `et####` case ID. Provides WHO consensus and original pathologist classification strings. We decode these to tissue codes and apply them as a fallback for eTumour rows not yet labelled via F2.

In [10]:
# Parse the pathology file
path_records = {}   # et_id → dict of raw fields
with open(DATA_DIR / FILES["F5"], "rb") as f:
    raw = f.read().decode("iso-8859-1")

for line in raw.splitlines():
    parts = [p.strip() for p in line.split("\t")]
    if not parts or not re.match(r"^et\d+", parts[0]):
        continue
    et_id = parts[0]
    consensus = parts[3] if len(parts) > 3 else ""
    original  = parts[5] if len(parts) > 5 else ""
    path_records[et_id] = {
        "centre":         parts[1] if len(parts) > 1 else "",
        "validated":      parts[2] if len(parts) > 2 else "",
        "who_consensus":  consensus,
        "who_original":   original,
        "tissue_consensus": who_to_code(consensus),
        "tissue_original":  who_to_code(original),
    }
    path_records[et_id]["tissue_code"] = (
        path_records[et_id]["tissue_consensus"]
        or path_records[et_id]["tissue_original"]
    )

file_usage["F5"].update({
    "loaded":  True,
    "records": len(path_records),
    "role":    "eTumour WHO pathology text → tissue code (fallback)",
})
print(f"F5 loaded: {len(path_records)} eTumour case records")
labelled_in_f5 = sum(1 for v in path_records.values() if v["tissue_code"])
print(f"  With resolvable tissue code: {labelled_in_f5}")
print(f"  consensus=NA and original=NA: {sum(1 for v in path_records.values() if not v['tissue_consensus'] and not v['tissue_original'])}")

# Apply to unlabelled eTumour rows
et_unlabelled_before_f5 = sum(
    1 for r in registry
    if r["dataset"] == "eTumour" and r["tissue_code"] is None
)
assigned_f5 = 0
for r in registry:
    if r["dataset"] != "eTumour" or r["tissue_code"] is not None:
        continue
    rec = path_records.get(r["case_id"])
    if rec and rec["tissue_code"]:
        r["tissue_code"]   = rec["tissue_code"]
        r["tissue_source"] = "F5_pathology_text"
        r["who_consensus"] = rec["who_consensus"]
        r["who_original"]  = rec["who_original"]
        if "F5" not in r["source_files"]: r["source_files"].append("F5")
        assigned_f5 += 1
    elif rec:
        # Case IS in F5 but no tissue could be assigned — record raw strings
        r["who_consensus"] = rec["who_consensus"]
        r["who_original"]  = rec["who_original"]
        if "F5" not in r["source_files"]: r["source_files"].append("F5")

print(f"\n  eTumour rows unlabelled before F5: {et_unlabelled_before_f5}")
print(f"  Newly labelled via F5:              {assigned_f5}")

# Remaining unlabelled eTumour — show their WHO strings for inspection
still_unlabelled_et = [
    r for r in registry
    if r["dataset"] == "eTumour" and r["tissue_code"] is None
]
print(f"  Still unlabelled eTumour rows: {len(still_unlabelled_et)}")

# Classify why each is still unlabelled
reasons = Counter()
for r in still_unlabelled_et:
    rec = path_records.get(r["case_id"], {})
    cons = str(rec.get("who_consensus", "")).strip()
    orig = str(rec.get("who_original",  "")).strip()
    if not rec:
        reasons["not_in_F5"] += 1
    elif cons in ("","NA","0") and orig in ("","NA","0","null","NULL"):
        reasons["both_NA_in_F5"] += 1
    elif cons == "OTHER" and orig in ("OTHER", "", "NA"):
        reasons["labelled_OTHER"] += 1
    else:
        reasons[f"unmapped_WHO_text"] += 1

print(f"  Reasons for remaining unlabelled:")
for reason, n in reasons.items():
    print(f"    {reason}: {n}")

# Full table of unmapped WHO strings for extension
unmapped_who = [
    {"case_id": r["case_id"], "echo_time": r["echo_time"],
     "quality": r["quality"],
     "who_consensus": r.get("who_consensus",""),
     "who_original":  r.get("who_original","")}
    for r in still_unlabelled_et
    if not (str(path_records.get(r["case_id"],{}).get("who_consensus","")).strip() in ("","NA","0")
            and str(path_records.get(r["case_id"],{}).get("who_original","")).strip() in ("","NA","0","null","OTHER"))
]
print(f"\n  Rows with unmapped WHO text (extend WHO_TO_TISSUE to recover):")
print(pd.DataFrame(unmapped_who).drop_duplicates(subset="case_id").to_string(index=False))

file_usage["F5"]["notes"] = (
    f"Labelled {assigned_f5} additional eTumour rows; "
    f"{len(still_unlabelled_et)} remain unlabelled after F2+F5"
)

F5 loaded: 1699 eTumour case records
  With resolvable tissue code: 1252
  consensus=NA and original=NA: 447

  eTumour rows unlabelled before F5: 843
  Newly labelled via F5:              740
  Still unlabelled eTumour rows: 103
  Reasons for remaining unlabelled:
    both_NA_in_F5: 79
    labelled_OTHER: 10
    unmapped_WHO_text: 14

  Rows with unmapped WHO text (extend WHO_TO_TISSUE to recover):
case_id echo_time quality who_consensus     who_original
 et2389     short     Bad         OTHER            OTHER
 et2969     short    Good         OTHER            OTHER
 et3119     short    Good         OTHER            OTHER
 et3125     short    Good            NA           STROKE
 et3370     short    Good         OTHER            OTHER
 et3385     short    Good            NA CHONDROMA 9220/0
 et3422     short     Bad         OTHER            OTHER
 et3559     short    Good         OTHER            OTHER
 et3749     short    Good         OTHER            OTHER


---
## 5.  F6 — `Listado_etumour-SNR-WBW.txt` (eTumour SNR and WBW)

Joins on `(et_id, experiment_index)`. Provides the DB-standard SNR and WBW values (the MAT `snr_mat` column uses a different definition). Only applies to eTumour rows.

In [11]:
snr_lookup = {}  # (et_id, exp_idx) → {snr_db, wbw_hz}
with open(DATA_DIR / FILES["F6"], "r", encoding="ascii", errors="replace") as f:
    for line in f:
        parts = [p.strip() for p in line.split("\t")]
        if not parts or not re.match(r"^et\d+", parts[0]):
            continue
        exp_m = re.search(r"idf(\d+)", parts[2]) if len(parts) > 2 else None
        exp_n = int(exp_m.group(1)) if exp_m else -1
        try:
            snr = float(parts[4]) if len(parts) > 4 and parts[4] else np.nan
            wbw = float(parts[5]) if len(parts) > 5 and parts[5] else np.nan
        except ValueError:
            snr, wbw = np.nan, np.nan
        snr_lookup[(parts[0], exp_n)] = {"snr_db": snr, "wbw_hz": wbw}

file_usage["F6"].update({
    "loaded": True, "records": len(snr_lookup),
    "role": "eTumour DB-standard SNR + WBW per spectrum",
})
print(f"F6 loaded: {len(snr_lookup)} (et_id, exp_idx) entries")

matched_f6 = unmatched_f6 = 0
for r in registry:
    if r["dataset"] != "eTumour":
        continue
    key = (r["case_id"], r["exp_idx"])
    entry = snr_lookup.get(key)
    if entry:
        r["snr_db"] = entry["snr_db"]
        r["wbw_hz"] = entry["wbw_hz"]
        if "F6" not in r["source_files"]: r["source_files"].append("F6")
        matched_f6 += 1
    else:
        unmatched_f6 += 1

print(f"  Matched to F1 eTumour rows: {matched_f6}")
print(f"  Unmatched eTumour rows:     {unmatched_f6}")

# Note: MAT snr_mat vs DB snr_db are different SNR definitions — show a comparison
both_snr = [
    (r["snr_mat"], r["snr_db"])
    for r in registry
    if r["dataset"] == "eTumour" and r["snr_db"] is not None
    and not np.isnan(r["snr_db"]) and r["snr_mat"] > 0
]
if both_snr:
    mat_vals = [x[0] for x in both_snr]
    db_vals  = [x[1] for x in both_snr]
    corr = np.corrcoef(mat_vals, db_vals)[0, 1]
    print(f"\n  SNR comparison (MAT col vs DB file):")
    print(f"    MAT snr: mean={np.mean(mat_vals):.1f}, range {min(mat_vals):.1f}-{max(mat_vals):.1f}")
    print(f"    DB  snr: mean={np.mean(db_vals):.1f}, range {min(db_vals):.1f}-{max(db_vals):.1f}")
    print(f"    Pearson correlation: {corr:.3f}")
    print(f"    (Different definitions — use DB snr_db for analysis)")

file_usage["F6"]["notes"] = f"Matched {matched_f6}/{matched_f6+unmatched_f6} eTumour rows; {unmatched_f6} unmatched"

F6 loaded: 1700 (et_id, exp_idx) entries
  Matched to F1 eTumour rows: 1371
  Unmatched eTumour rows:     0

  SNR comparison (MAT col vs DB file):
    MAT snr: mean=13.6, range 0.1-78.8
    DB  snr: mean=54.8, range 0.0-1249.6
    Pearson correlation: 0.401
    (Different definitions — use DB snr_db for analysis)


---
## 6.  F7 + F8 — XLS files (inspect only)

These are legacy `.xls` files that require `xlrd`. We inspect their structure and record what they contain. They are metadata tables, not spectrum sources, so no registry rows are generated.

In [14]:
try:
    import xlrd
    for fid, fname in [("F7", FILES["F7"]), ("F8", FILES["F8"])]:
        path = DATA_DIR / fname
        wb = xlrd.open_workbook(str(path))
        print(f"{fid} ({fname}):")
        for sh in wb.sheets():
            print(f"  Sheet '{sh.name}': {sh.nrows} rows × {sh.ncols} cols")
            # Print header row
            if sh.nrows > 0:
                headers = [str(sh.cell(0, c).value) for c in range(min(sh.ncols, 8))]
                print(f"  Headers: {headers}")
            # Print sample row
            if sh.nrows > 1:
                sample = [str(sh.cell(1, c).value) for c in range(min(sh.ncols, 8))]
                print(f"  Row 1:   {sample}")
        file_usage[fid].update({"loaded": True, "records": sum(s.nrows for s in wb.sheets()),
            "role": "Pathology metadata table (inspect only — no spectra)",
            "notes": f"{wb.nsheets} sheet(s)"})
except ImportError:
    print("xlrd not installed — install with: pip install xlrd==1.2.0")
    print("Skipping F7 and F8 inspection.")
    for fid in ["F7", "F8"]:
        file_usage[fid]["notes"] = "Skipped — xlrd not installed"
except Exception as e:
    print(f"Error reading XLS files: {e}")

F7 (Nocaso_patologÍa_base_de_datoS.xls):
  Sheet 'Hoja2': 2005 rows × 11 cols
  Headers: ['', 'CENTRE ID', 'CR HISTOPATHOLOGY VALIDATED', 'CR PARAFFIN SECTION WHO CLASSIFICATION (CONSENSUS)', 'CR PARAFFIN SECTION WHO CLASSIFICATION (INTERNAL PANEL OF PATHOLOGISTS)', 'CR PARAFFIN SECTION WHO CLASSIFICATION (ORIGINATING PATHOLOGIST)', '', '']
  Row 1:   ['et2001 ', 'MEDICAL UNIVERSITY OF LODZ ', 'NA ', 'NA ', 'NA ', 'ATYPICAL MENINGIOMA 9539/1 ', '', '']
  Sheet 'Hoja1': 1181 rows × 8 cols
  Headers: ['', '', '', '', '', '', '', '']
  Row 1:   ['0.0', 'et2024', 'glioblastoma', "'Good'", '', '', '', '']
  Sheet 'Hoja3': 0 rows × 0 cols
F8 (Tablas_consensus_vs_original_pathology.xls):
  Sheet 'Tablas_resumen': 40 rows × 11 cols
  Headers: ['STE  eTUMOUR and INTERPRET', 'Spectral quality', '', '', 'Clinical quality (diagnostic)', '', '', '']
  Row 1:   ['tumour type', 'GOOD/Consensus', 'BAD/Consensus pathology', 'POOR/Consensus', 'Consensus', '', '', 'GOOD/Consensus pathology']
  Sheet 'Hoj

In [13]:
!pip install xlrd==1.2.0


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 7.  F9 / F10 / F11 — Reconstructed spectrum XMLs

These are artefact-removed versions of the original spectra. For the artefact *generation* project we want originals, not cleaned ones. Here we:
1. Load each file and record case counts and tissue distributions.
2. Cross-match against F2 (original XML) to confirm they are reconstructions of the same cases.
3. Mark each as a duplicate-of-F2 or orphan in the audit.

In [15]:
recon_files = {
    "F9":  (FILES["F9"],  "K=20, 17 sources"),
    "F10": (FILES["F10"], "K=5, 16 sources"),
    "F11": (FILES["F11"], "K=20, 8 sources"),
}

# Reference: F2 original XML spectra
f2_ids = [r["case_id"] for r in f2_records]

recon_audit_rows = []

for fid, (fname, description) in recon_files.items():
    records = parse_xml_cases(DATA_DIR / fname)
    matrix  = spectrum_matrix(records)
    rec_ids = [r["case_id"] for r in records]

    print(f"\n{fid} ({fname}) — {description}")
    print(f"  Cases: {len(records)}")
    tc_dist = Counter(r["tissue_code"] for r in records)
    print(f"  Tissue dist: {dict(tc_dist.most_common(6))} ...")

    # Cross-match Ori_N IDs against F2 IDs
    id_overlap  = len(set(rec_ids) & set(f2_ids))
    id_only_here = len(set(rec_ids) - set(f2_ids))
    id_only_f2   = len(set(f2_ids) - set(rec_ids))
    print(f"  ID overlap with F2 (Matriz_originales): {id_overlap}")
    print(f"  IDs only in {fid}: {id_only_here}")
    print(f"  IDs only in F2:  {id_only_f2}")

    # Spectrum-level check: pick 10 random reconstructed spectra,
    # find nearest original in F2 — expect near-match (same case, cleaned)
    sample_n = min(10, len(records))
    sample_indices = np.random.choice(len(records), sample_n, replace=False)
    near_count = 0
    sample_mses = []
    for si in sample_indices:
        _, mse = find_best_match(records[si]["spectrum"], f2_matrix)
        sample_mses.append(mse)
        if mse < THRESH_NEAR:
            near_count += 1
    print(f"  Spectrum similarity to F2 (sample {sample_n}):")
    print(f"    Min MSE={min(sample_mses):.4f}, Max MSE={max(sample_mses):.4f}, Mean={np.mean(sample_mses):.4f}")
    print(f"    Near-matches (MSE<{THRESH_NEAR}): {near_count}/{sample_n}")
    print(f"  → These are CLEANED reconstructions — do NOT use as artefact training data")

    file_usage[fid].update({
        "loaded": True, "records": len(records),
        "role": f"Reconstructed (artefact-removed) spectra — {description}",
        "notes": f"Derived from F2; NOT used in artefact training dataset",
    })

    recon_audit_rows.append({
        "file_id": fid, "description": description,
        "n_cases": len(records),
        "f2_id_overlap": id_overlap,
        "f2_only_here": id_only_here,
        "f2_only_f2": id_only_f2,
        "sample_mean_mse": round(np.mean(sample_mses), 4),
        "verdict": "cleaned_reconstructions_of_F2_not_used",
    })

print(f"\n" + "─"*60)
print("Reconstructed files audit summary:")
print(pd.DataFrame(recon_audit_rows).to_string(index=False))


F9 (Matrices_Yeni_XML/Matriz_Reconstruido_K20_17_2018.xml) — K=20, 17 sources
  Cases: 699
  Tissue dist: {'gl': 292, 'mm': 121, 'me': 116, 'a2': 103, 'od': 44, 'oa': 23} ...
  ID overlap with F2 (Matriz_originales): 0
  IDs only in F9: 699
  IDs only in F2:  775
  Spectrum similarity to F2 (sample 10):
    Min MSE=0.2525, Max MSE=4.1180, Mean=1.0396
    Near-matches (MSE<0.5): 3/10
  → These are CLEANED reconstructions — do NOT use as artefact training data

F10 (Matrices_Yeni_XML/Matriz_Reconstruidos_K5_16_2018_1.xml) — K=5, 16 sources
  Cases: 801
  Tissue dist: {'gl': 292, 'mm': 121, 'me': 116, 'a2': 103, 'pi': 54, 'od': 44} ...
  ID overlap with F2 (Matriz_originales): 0
  IDs only in F10: 801
  IDs only in F2:  775
  Spectrum similarity to F2 (sample 10):
    Min MSE=0.3492, Max MSE=4.9062, Mean=1.2219
    Near-matches (MSE<0.5): 2/10
  → These are CLEANED reconstructions — do NOT use as artefact training data

F11 (Matrices_Yeni_XML/Matriz_Reconstruidos_K20_8sources.xml) — K=20

---
## 8.  F12 / F13 / F14 / F15 — `short.mat`, `Long_echo_data.mat` and label files

These contain spectra and quality labels but **no metadata columns** (no case ID, SNR, WBW). We check whether their spectra are exact duplicates of rows already in F1, and whether their quality labels agree.

In [16]:
f12 = sio.loadmat(DATA_DIR / FILES["F12"])
f13 = sio.loadmat(DATA_DIR / FILES["F13"])
f14 = sio.loadmat(DATA_DIR / FILES["F14"])
f15 = sio.loadmat(DATA_DIR / FILES["F15"])

short_only_specs  = f12["Short"].astype(np.float32)     # (N, 512)
long_only_specs   = f13["LONG"].astype(np.float32)      # (N, 512)
short_only_labels = np.array([str(l.flat[0]) for l in f14["ShortTE_label"].flatten()])
long_only_labels  = np.array([str(l.flat[0]) for l in f15["LongTE_label"].flatten()])

for fid, arr, role in [
    ("F12", short_only_specs,  "short-TE spectra only"),
    ("F13", long_only_specs,   "long-TE spectra only"),
    ("F14", short_only_labels, "quality labels for F12"),
    ("F15", long_only_labels,  "quality labels for F13"),
]:
    file_usage[fid].update({"loaded": True, "records": len(arr), "role": role})

print(f"F12 (short.mat):           {short_only_specs.shape}")
print(f"F13 (Long_echo_data.mat):  {long_only_specs.shape}")
print(f"F14 (label.mat):           {len(short_only_labels)} labels: {dict(Counter(short_only_labels))}")
print(f"F15 (Long_echo_labels.mat):{len(long_only_labels)} labels: {dict(Counter(long_only_labels))}")

# Build reference matrices from F1 (the known good source)
f1_short_specs = np.vstack([r["spectrum"] for r in registry if r["echo_time"] == "short"])
f1_long_specs  = np.vstack([r["spectrum"] for r in registry if r["echo_time"] == "long"])
f1_short_qs    = [r["quality"] for r in registry if r["echo_time"] == "short"]
f1_long_qs     = [r["quality"] for r in registry if r["echo_time"] == "long"]

print(f"\nMatching F12/F13 spectra against F1...")

for label, standalone_specs, standalone_labels, f1_specs, f1_qs, fspec, flab in [
    ("short", short_only_specs, short_only_labels, f1_short_specs, f1_short_qs, "F12", "F14"),
    ("long",  long_only_specs,  long_only_labels,  f1_long_specs,  f1_long_qs,  "F13", "F15"),
]:
    exact_match_count    = 0
    label_agree_count    = 0
    label_disagree_count = 0
    no_match_count       = 0
    disagree_examples    = []

    for i in range(len(standalone_specs)):
        best_idx, best_mse = find_best_match(standalone_specs[i], f1_specs)
        if best_mse < THRESH_EXACT:
            exact_match_count += 1
            if standalone_labels[i] == f1_qs[best_idx]:
                label_agree_count += 1
            else:
                label_disagree_count += 1
                if len(disagree_examples) < 5:
                    disagree_examples.append({
                        f"{fspec}_row": i,
                        f"{fspec}_label": standalone_labels[i],
                        "F1_label":      f1_qs[best_idx],
                        "mse":           round(best_mse, 8),
                    })
        else:
            no_match_count += 1

    print(f"\n  {label.upper()}-TE ({fspec}/{flab}): {len(standalone_specs)} rows")
    print(f"    Exact match in F1 (MSE<{THRESH_EXACT}): {exact_match_count}")
    print(f"    Label agrees with F1:                   {label_agree_count}")
    print(f"    Label DISAGREES with F1:                {label_disagree_count}")
    print(f"    No exact match in F1:                   {no_match_count}")
    if disagree_examples:
        print(f"    Disagreement examples:")
        print(pd.DataFrame(disagree_examples).to_string(index=False))

    verdict = (
        "exact_duplicate_of_F1" if no_match_count == 0
        else f"partial_duplicate_{exact_match_count}_of_{len(standalone_specs)}_match"
    )
    file_usage[fspec]["notes"] = verdict
    file_usage[flab]["notes"]  = f"Label source for {fspec}; {label_disagree_count} label disagreements with F1"

print("\nConclusion: F12/F13/F14/F15 are subsets of F1 if exact_match_count == total rows.")

F12 (short.mat):           (1180, 512)
F13 (Long_echo_data.mat):  (977, 512)
F14 (label.mat):           1180 labels: {np.str_('Good'): 982, np.str_('Poor'): 49, np.str_('Bad'): 149}
F15 (Long_echo_labels.mat):977 labels: {np.str_('Good'): 828, np.str_('Bad'): 111, np.str_('Poor'): 38}

Matching F12/F13 spectra against F1...

  SHORT-TE (F12/F14): 1180 rows
    Exact match in F1 (MSE<0.0001): 1180
    Label agrees with F1:                   1180
    Label DISAGREES with F1:                0
    No exact match in F1:                   0

  LONG-TE (F13/F15): 977 rows
    Exact match in F1 (MSE<0.0001): 977
    Label agrees with F1:                   977
    Label DISAGREES with F1:                0
    No exact match in F1:                   0

Conclusion: F12/F13/F14/F15 are subsets of F1 if exact_match_count == total rows.


---
## 9.  F16 — `convex_nmf.py`

Algorithm source file, not a data file. Inspect for usability (Python 2 vs 3 compatibility).

In [17]:
with open(DATA_DIR / FILES["F16"], "r", errors="replace") as f:
    code = f.read()

py2_prints = re.findall(r"^\s*print [^(]", code, re.MULTILINE)
print(f"F16 ({FILES['F16']}):")
print(f"  Lines: {len(code.splitlines())}")
print(f"  Python 2 print statements: {len(py2_prints)}")
print(f"  Status: {'NEEDS PORTING to Python 3' if py2_prints else 'Python 3 compatible'}")

file_usage["F16"].update({
    "loaded": True, "records": 0,
    "role": "Convex-NMF algorithm (Python 2)",
    "notes": f"{len(py2_prints)} Python 2 print statements — port before running",
})

F16 (Matrices_y_datos_tesis_Yeni/convex_nmf.py):
  Lines: 129
  Python 2 print statements: 2
  Status: NEEDS PORTING to Python 3


---
## 10.  Build the master registry DataFrame and label audit

In [18]:
# Convert registry list to DataFrame (drop raw spectrum arrays — keep separately)
meta_cols = [
    "row_uid", "case_id", "dataset", "echo_time", "exp_idx",
    "quality", "quality_code", "is_artefactual", "quest_rmse",
    "snr_mat", "wbw_mat", "snr_db", "wbw_hz",
    "tissue_code", "tissue_source", "who_consensus", "who_original",
    "xml_match_file", "xml_match_id", "xml_match_mse",
]

df = pd.DataFrame([
    {col: r.get(col) for col in meta_cols} | {"source_files": "|".join(r["source_files"])}
    for r in registry
])

# Spectrum matrix (aligned to df row order)
spectra = np.vstack([r["spectrum"] for r in registry]).astype(np.float32)

print(f"Master registry: {len(df)} rows × {len(df.columns)} columns")
print(f"Spectra matrix:  {spectra.shape}")

# ── Summary ───────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("LABEL COVERAGE SUMMARY")
print("=" * 60)

for te in ["short", "long"]:
    sub = df[df["echo_time"] == te]
    labelled = sub["tissue_code"].notna()
    print(f"\n{te.upper()}-TE  (n={len(sub)})")
    print(f"  Tissue-labelled:   {labelled.sum()} ({100*labelled.mean():.1f}%)")
    print(f"  Unlabelled:        {(~labelled).sum()}")
    print(f"  Quality: {dict(sub['quality'].value_counts())}")
    print(f"  Tissue source breakdown:")
    print(sub["tissue_source"].value_counts().to_string())

print(f"\n{'─'*60}")
print("Unlabelled rows by dataset:")
unlabelled = df[df["tissue_code"].isna()]
print(unlabelled.groupby(["dataset", "echo_time"])["quality"].value_counts().to_string())

Master registry: 2157 rows × 21 columns
Spectra matrix:  (2157, 512)

LABEL COVERAGE SUMMARY

SHORT-TE  (n=1180)
  Tissue-labelled:   990 (83.9%)
  Unlabelled:        190
  Quality: {np.str_('Good'): np.int64(982), np.str_('Bad'): np.int64(149), np.str_('Poor'): np.int64(49)}
  Tissue source breakdown:
tissue_source
F2_exact_match       528
F3_or_F4_id_match    298
F5_pathology_text    164

LONG-TE  (n=977)
  Tissue-labelled:   816 (83.5%)
  Unlabelled:        161
  Quality: {np.str_('Good'): np.int64(828), np.str_('Bad'): np.int64(111), np.str_('Poor'): np.int64(38)}
  Tissue source breakdown:
tissue_source
F5_pathology_text    576
F3_or_F4_id_match    240

────────────────────────────────────────────────────────────
Unlabelled rows by dataset:
dataset    echo_time  quality
INTERPRET  long       Good       96
                      Bad        16
                      Poor        4
           short      Good       93
                      Bad        34
                      Poor        

---
## 11.  File-level audit: used, unused, duplicates

In [19]:
print("=" * 80)
print("FILE AUDIT — USAGE SUMMARY")
print("=" * 80)

audit_rows = []
for fid, fname in FILES.items():
    info = file_usage[fid]
    present = (DATA_DIR / fname).exists()
    audit_rows.append({
        "File ID":    fid,
        "File name":  fname,
        "Present":    "Yes" if present else "NO — MISSING",
        "Loaded":     "Yes" if info["loaded"] else "No",
        "Records":    info["records"],
        "Role":       info["role"],
        "Notes":      info["notes"],
    })

audit_df = pd.DataFrame(audit_rows)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
print(audit_df.to_string(index=False))

# Source file count per registry row
print(f"\n{'─'*60}")
print("Registry rows by number of source files contributing:")
source_counts = df["source_files"].str.split("|").apply(len).value_counts().sort_index()
for n_files, count in source_counts.items():
    print(f"  {n_files} file(s): {count} rows")

print(f"\n{'─'*60}")
print("Which files appear in source_files column:")
for fid in ["F1","F2","F3","F4","F5","F6"]:
    count = df["source_files"].str.contains(fid).sum()
    print(f"  {fid}: {count} rows")

print(f"\n{'─'*60}")
print("Files NOT contributing any data to the final registry:")
for fid in ["F7","F8","F9","F10","F11","F12","F13","F14","F15","F16"]:
    print(f"  {fid} ({FILES[fid]}): {file_usage[fid]['notes']}")

FILE AUDIT — USAGE SUMMARY
File ID                                               File name Present Loaded  Records                                                        Role                                                                                                                            Notes
     F1   Matrices_y_datos_tesis_Yeni/Multiclass_RMSE_QUEST.mat     Yes    Yes     2157                     Primary spectrum + quality label source                                                                                                                                 
     F2                 Matrices_Yeni_XML/Matriz_originales.xml     Yes    Yes      777              eTumour tissue labels via exact spectrum match 535 XML spectra exactly matched to F1 rows; 242 XML spectra unmatched (different preprocessing); 19 near-matches (0 < MSE < 0.5)
     F3 Matrices_y_datos_tesis_Yeni/1TE-ShortMRS(corregido).xml     Yes    Yes      304                      INTERPRET tissue labels (short-TE

---
## 12.  Export results

In [20]:
# ── 1. Master metadata CSV ─────────────────────────────────────────────────
df.to_csv("mrs_registry_metadata.csv", index=False)
print(f"Saved: mrs_registry_metadata.csv  ({len(df)} rows)")

# ── 2. Spectra matrix ──────────────────────────────────────────────────────
np.save("mrs_registry_spectra.npy", spectra)
print(f"Saved: mrs_registry_spectra.npy   {spectra.shape}")

# ── 3. Unlabelled detail CSV ───────────────────────────────────────────────
unlabelled_df = df[df["tissue_code"].isna()].copy()
unlabelled_df.drop(columns=["spectrum"] if "spectrum" in unlabelled_df.columns else []).to_csv(
    "mrs_unlabelled_detail.csv", index=False
)
print(f"Saved: mrs_unlabelled_detail.csv  ({len(unlabelled_df)} rows)")

# ── 4. INTERPRET unresolved spectrum match results ─────────────────────────
if len(interp_spec_match_df):
    interp_spec_match_df.drop(columns=["reg_idx"]).to_csv(
        "mrs_interp_spectrum_matches.csv", index=False
    )
    print(f"Saved: mrs_interp_spectrum_matches.csv ({len(interp_spec_match_df)} rows)")

# ── 5. File audit CSV ──────────────────────────────────────────────────────
audit_df.to_csv("mrs_file_audit.csv", index=False)
print(f"Saved: mrs_file_audit.csv")

# ── 6. F2 near-match table ─────────────────────────────────────────────────
if near_match_records:
    pd.DataFrame(near_match_records).to_csv("mrs_f2_near_matches.csv", index=False)
    print(f"Saved: mrs_f2_near_matches.csv ({len(near_match_records)} rows)")

print("\nAll done.")
print(f"  Primary dataset: {len(df)} spectra, {df['tissue_code'].notna().sum()} tissue-labelled ({100*df['tissue_code'].notna().mean():.1f}%)")
print(f"  Artefactual (Poor+Bad): {df['is_artefactual'].sum()} ({100*df['is_artefactual'].mean():.1f}%)")

Saved: mrs_registry_metadata.csv  (2157 rows)
Saved: mrs_registry_spectra.npy   (2157, 512)
Saved: mrs_unlabelled_detail.csv  (351 rows)
Saved: mrs_interp_spectrum_matches.csv (248 rows)
Saved: mrs_file_audit.csv
Saved: mrs_f2_near_matches.csv (19 rows)

All done.
  Primary dataset: 2157 spectra, 1806 tissue-labelled (83.7%)
  Artefactual (Poor+Bad): 347 (16.1%)


In [33]:
import pickle

with open(r"C:\\Users\\matth\\Documents\\PhD\\code\\notebooks\\yani_etumour_reconstruction\\data\\Matrices_y_datos_tesis_Yeni\\Reconstrucción\\resultsSTEF5.pkl", "rb") as f:
    print((pickle.load(f, encoding="latin1")).shape)

(512, 5)


In [41]:
import os
import xml.etree.ElementTree as ET

folder = r"C:\\Users\\matth\\Documents\\PhD\\code\\notebooks\\yani_etumour_reconstruction\\data"

for root_dir, _, files in os.walk(folder):
    for file in files:
        if file.endswith(".xml"):
            path = os.path.join(root_dir, file)
            try:
                tree = ET.parse(path)
                root = tree.getroot()
                dataset = root if root.tag == "DATASET" else root.find("DATASET")
                count = len(dataset.findall("Case")) if dataset is not None else 0
                print(f"{path}: {count}")
            except Exception:
                print(f"{path}: error")

C:\\Users\\matth\\Documents\\PhD\\code\\notebooks\\yani_etumour_reconstruction\\data\Matrices_Yeni_XML\Matriz_originales.xml: 777
C:\\Users\\matth\\Documents\\PhD\\code\\notebooks\\yani_etumour_reconstruction\\data\Matrices_Yeni_XML\Matriz_Reconstruidos_K20_8sources.xml: 699
C:\\Users\\matth\\Documents\\PhD\\code\\notebooks\\yani_etumour_reconstruction\\data\Matrices_Yeni_XML\Matriz_Reconstruidos_K5_16_2018_1.xml: 801
C:\\Users\\matth\\Documents\\PhD\\code\\notebooks\\yani_etumour_reconstruction\\data\Matrices_Yeni_XML\Matriz_Reconstruido_K20_17_2018.xml: 699
C:\\Users\\matth\\Documents\\PhD\\code\\notebooks\\yani_etumour_reconstruction\\data\Matrices_y_datos_tesis_Yeni\1TE-LongMRS(corregido).xml: 266
C:\\Users\\matth\\Documents\\PhD\\code\\notebooks\\yani_etumour_reconstruction\\data\Matrices_y_datos_tesis_Yeni\1TE-ShortMRS(corregido).xml: 304
C:\\Users\\matth\\Documents\\PhD\\code\\notebooks\\yani_etumour_reconstruction\\data\Matrices_y_datos_tesis_Yeni\Reconstrucción\Matriz_original

In [45]:
import os
import pickle

folder = r"C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data"

for root_dir, _, files in os.walk(folder):
    for file in files:
        if file.endswith((".pkl", ".pkl.bak")):
            path = os.path.join(root_dir, file)
            try:
                with open(path, "rb") as f:
                    obj = pickle.load(f, encoding="latin1")
                shape = getattr(obj, "shape", None)
                print(f"{path}: {shape}")
            except Exception:
                print(f"{path}: error")

C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Matrices_y_datos_tesis_Yeni\Fuentes extraidas LTE\resultsLTEM10.pkl: (10, 3)
C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Matrices_y_datos_tesis_Yeni\Fuentes extraidas LTE\resultsLTEM11.pkl: (10, 3)
C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Matrices_y_datos_tesis_Yeni\Fuentes extraidas LTE\resultsLTEM12.pkl: (10, 3)
C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Matrices_y_datos_tesis_Yeni\Fuentes extraidas LTE\resultsLTEM13.pkl: (10, 3)
C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Matrices_y_datos_tesis_Yeni\Fuentes extraidas LTE\resultsLTEM14.pkl: (10, 3)
C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Matrices_y_datos_tesis_Yeni\Fuentes extraidas LTE\resultsLTEM15.pkl: (10, 3)
C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data

In [46]:
import os
from scipy.io import loadmat

folder = r"C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data"

for root_dir, _, files in os.walk(folder):
    for file in files:
        if file.endswith(".mat"):
            path = os.path.join(root_dir, file)
            try:
                data = loadmat(path)
                shapes = {k: getattr(v, "shape", None) for k, v in data.items() if not k.startswith("__")}
                print(f"{path}: {shapes}")
            except Exception:
                print(f"{path}: error")

C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Matrices_y_datos_tesis_Yeni\label.mat: {'ShortTE_label': (1180, 1)}
C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Matrices_y_datos_tesis_Yeni\Long_echo_data.mat: {'LONG': (977, 512)}
C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Matrices_y_datos_tesis_Yeni\Long_echo_labels.mat: {'LongTE_label': (977, 1)}
C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Matrices_y_datos_tesis_Yeni\Multiclass_RMSE_QUEST.mat: {'RMSE_LongTE': (977, 4), 'LongTE': (977, 517), 'LongTE_label': (977, 1), 'ShortTE': (1180, 517), 'ShortTE_label': (1180, 1), 'RMSE_ShortTE': (1180, 4)}
C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Matrices_y_datos_tesis_Yeni\short.mat: {'Short': (1180, 512)}


In [49]:
import numpy as np
import re

path = r"C:\Users\matth\Documents\PhD\code\notebooks\yani_etumour_reconstruction\data\Listado_etumour2.txt"

with open(path, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

cases = re.findall(r"\bet\d+\b", text)
arr = np.array(cases)

print(arr.shape)

(3398,)
